In [1]:
!pip install -q segmentation_models_pytorch
import os, sys, subprocess
import random, warnings, rasterio, torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import albumentations as A
import segmentation_models_pytorch as smp
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data')
IMAGE_DIR, LABEL_DIR = BASE / 'image', BASE / 'label'
PRED_DIR, SPLIT_DIR = BASE / 'prediction/image', BASE / 'split'

NUM_CLASSES = 3   # 0: Non-flood, 1: Flood, 2: Water bodies [cite: 32-35]
NUM_CHANNELS = 11 # 6 Original + 5 Engineered
PATCH_SIZE = 512
BATCH_SIZE = 4    # Reduced for heavier encoder
EPOCHS = 40       # Increased for Transformer convergence
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Step 1: Advanced Geospatial Feature Engineering ──────────────────────────
def get_advanced_features(img):
    # Original: SAR (VV, VH), Optical (Green, Red, NIR, SWIR) [cite: 38]
    vv, vh = img[0], img[1]
    green, red, nir, swir = img[2], img[3], img[4], img[5]
    
    # Indices for better water discrimination 
    ndwi = ((green - nir) / (green + nir + 1e-8))[np.newaxis]
    mndwi = ((green - swir) / (green + swir + 1e-8))[np.newaxis] # Better for urban
    ndvi = ((nir - red) / (nir + red + 1e-8))[np.newaxis]
    ndmi = ((nir - swir) / (nir + swir + 1e-8))[np.newaxis]
    sar_ratio = (vv / (vh + 1e-8))[np.newaxis]

    feats = np.concatenate([ndwi, mndwi, ndvi, ndmi, sar_ratio], axis=0)
    # Clip to standard range and concatenate
    return np.concatenate([img, np.clip(feats, -1, 1)], axis=0)

# ── Step 2: Corrected Dataset with Length and Modern Transforms ──────────────
class AISEHackDataset(Dataset):
    def __init__(self, ids, image_dir, label_dir=None, augment=False):
        self.ids = ids
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        
        # Switched ShiftScaleRotate to Affine per the UserWarning
        self.augment = A.Compose([
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.06), rotate=(-15, 15), p=0.5),
            A.OneOf([
                A.GridDistortion(p=0.5),
                A.OpticalDistortion(p=0.5),
            ], p=0.3),
        ]) if augment else None

    # This was the missing function causing the TypeError
    def __len__(self): 
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        with rasterio.open(self.image_dir / f'{sid}_image.tif') as src:
            # Applying feature engineering [cite: 38]
            img = get_advanced_features(src.read().astype(np.float32))
        
        # Channel-wise Normalization for all 11 channels
        for i in range(img.shape[0]):
            img[i] = (img[i] - np.mean(img[i])) / (np.std(img[i]) + 1e-8)

        if self.label_dir:
            with rasterio.open(self.label_dir / f'{sid}_label.tif') as src:
                lbl = src.read(1).astype(np.int64)
            # Clip and handle NaNs to ensure 3 classes (0, 1, 2) [cite: 32-35]
            lbl = np.clip(np.nan_to_num(lbl, nan=0), 0, NUM_CLASSES-1)
            if self.augment:
                # Transpose for Albumentations (H, W, C)
                aug = self.augment(image=img.transpose(1,2,0), mask=lbl)
                img, lbl = aug['image'].transpose(2,0,1), aug['mask']
            return torch.from_numpy(img).float(), torch.from_numpy(lbl).long()
        
        return torch.from_numpy(img).float(), sid
      

# ── Step 3: Architecture & Hybrid Loss ───────────────────────────────────────
def build_model():
    # MaxViT Tiny offers transformer power with convolutional efficiency
    model = smp.UnetPlusPlus(
        encoder_name="tu-maxvit_tiny_tf_512", 
        encoder_weights="imagenet",
        in_channels=NUM_CHANNELS, 
        classes=NUM_CLASSES,
        decoder_attention_type="scse" # Spatial & Channel Squeeze/Excite
    ).to(DEVICE)
    return model

class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='multiclass')
        self.focal = smp.losses.FocalLoss(mode='multiclass')

    def forward(self, y_pr, y_gt):
        # Combines Dice (IoU focus) with Focal (hard-pixel focus)
        return 0.4 * self.dice(y_pr, y_gt) + 0.6 * self.focal(y_pr, y_gt)

# ── Step 4: Training & TTA Inference ─────────────────────────────────────────
def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

if __name__ == "__main__":
    def load_ids(f): return [l.strip() for l in open(SPLIT_DIR / f) if l.strip()]
    train_ids, pred_ids = load_ids('train.txt'), load_ids('pred.txt')

    loader = DataLoader(AISEHackDataset(train_ids, IMAGE_DIR, LABEL_DIR, True), 
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    
    model = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, 
                                              steps_per_epoch=len(loader), epochs=EPOCHS)
    criterion = HybridLoss()
    scaler = GradScaler()

    # Training
    for epoch in range(EPOCHS):
        model.train()
        for imgs, masks in tqdm(loader, desc=f'Epoch {epoch+1}'):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

    # Inference with 4-way TTA
    model.eval()
    results = []
    with torch.no_grad():
        for imgs, sids in tqdm(DataLoader(AISEHackDataset(pred_ids, PRED_DIR), batch_size=1), desc='TTA Inference'):
            imgs = imgs.to(DEVICE)
            with autocast():
                # Augmentations: Original, H-Flip, V-Flip, HV-Flip
                p1 = torch.softmax(model(imgs), dim=1)
                p2 = torch.flip(torch.softmax(model(torch.flip(imgs, [3])), dim=1), [3])
                p3 = torch.flip(torch.softmax(model(torch.flip(imgs, [2])), dim=1), [2])
                p4 = torch.flip(torch.softmax(model(torch.flip(imgs, [2, 3])), dim=1), [2, 3])
                
                final_prob = (p1 + p2 + p3 + p4) / 4
                mask = final_prob.argmax(dim=1).cpu().numpy()[0]
            
            # Encode ONLY the 'Flood' class (Class 1) [cite: 45]
            results.append({'id': sids[0], 'rle_mask': rle_encode((mask == 1).astype(np.uint8))})

    pd.DataFrame(results).to_csv('submission.csv', index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.6 MB/s eta 0:00:00


model.safetensors:   0%|          | 0.00/124M [00:00<?, ?B/s]

/tmp/ipykernel_24/3109949596.py:133: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/15 [00:00<?, ?it/s]/tmp/ipykernel_24/3109949596.py:141: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
TTA Inference:   0%|          | 0/19 [00:00<?, ?it/s]/tmp/ipykernel_24/3109949596.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
TTA Inference: 100%|██████████| 19/19 [00:09<00:00,  1.95it/s]
